In [21]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table

In [22]:
URL_DATA = 'https://storage.dosm.gov.my/labour/lfs_state_sex.parquet'
df = pd.read_parquet(URL_DATA)


In [23]:
df.head()

,state,sex,date,lf,lf_employed,lf_unemployed,lf_outside,p_rate,u_rate,ep_ratio
0,Johor,both,1982-01-01,653.1,630.2,22.9,345.1,65.4,3.5,63.4
1,Johor,both,1983-01-01,650.9,623.7,27.2,344.1,65.4,4.2,60.5
2,Johor,both,1984-01-01,688.2,657.3,30.9,372.1,64.9,4.5,61.7
3,Johor,both,1985-01-01,710.4,672.7,37.6,354.7,66.7,5.3,61.2
4,Johor,both,1986-01-01,751.2,700.4,50.8,367.8,67.1,6.8,61.7


In [24]:
df["date"] = pd.to_datetime(df["date"])

df = (
    df[
        (df["date"] >= "2016-01-01") &
        (df["sex"] == "both") 
    ]
    .copy()
)

df = df.drop(columns=["sex", "p_rate"])

In [25]:
df.rename(columns={
    "lf": "total_labour",
    "lf_employed": "total_emp",
    "lf_unemployed": "total_unemp",
    "lf_outside": "total_olf",
    "u_rate": "lab_unemp_rate",
    "ep_ratio": "employment_pop_ratio"
}, inplace=True)

num_cols = [c for c in df.columns if c not in ['state', 'date']]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    if col.endswith('_rate') or col.endswith('_ratio'):
        continue
    df[col] = df[col] * 1000

In [26]:
df.head()

,state,date,total_labour,total_emp,total_unemp,total_olf,lab_unemp_rate,employment_pop_ratio
32,Johor,2016-01-01,1639100.0,1580600.0,58500.0,820700.0,3.6,62.5
33,Johor,2017-01-01,1673800.0,1616700.0,57100.0,824400.0,3.4,63.0
34,Johor,2018-01-01,1745100.0,1693300.0,51900.0,788200.0,3.0,65.0
35,Johor,2019-01-01,1805700.0,1756100.0,49600.0,761600.0,2.7,67.3
36,Johor,2020-01-01,1990900.0,1920500.0,70300.0,826900.0,3.5,68.4


In [27]:
write_table(df, "sc_bronze", "datagov_labour")

Table sc_bronze.datagov_labour written successfully.
